# 02 — EDA

Length distributions, source breakdown, class balance, vocabulary stats.

**Runtime:** CPU.

In [ ]:
import os, sys
os.chdir('/content/polygence'); sys.path.insert(0, 'src')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
from prompt_classifier.config import load_config

cfg = load_config('configs/base.yaml')
train = pd.read_parquet(cfg['data']['train_path'])
val   = pd.read_parquet(cfg['data']['val_path'])
test  = pd.read_parquet(cfg['data']['test_path'])
unified = pd.concat([train, val, test], ignore_index=True)
print(f'Total: {len(unified):,}  |  block={( unified.y==1).sum():,}  safe={(unified.y==0).sum():,}')

In [ ]:
# Per-source breakdown
print(unified.groupby(['source', 'y']).size().unstack(fill_value=0))

In [ ]:
# Prompt length distribution by label
unified['char_len'] = unified['prompt'].str.len()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, name) in zip(axes, [(0, 'Safe'), (1, 'Block')]):
    subset = unified[unified.y == label]['char_len']
    ax.hist(subset.clip(upper=subset.quantile(0.99)), bins=50, edgecolor='black')
    ax.set_title(f'{name} prompt length (chars)')
    ax.set_xlabel('chars')
plt.tight_layout(); plt.show()

In [ ]:
# Token length estimate (whitespace tokenization as proxy)
unified['word_len'] = unified['prompt'].str.split().str.len()
print(unified.groupby('y')['word_len'].describe())